### Siying Li
### Take Home Exercise 2

In [0]:
df_pit = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True)

In [0]:
display(df_pit)

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)

In [0]:
display(df_drivers)

### Problem1: What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

####### Logic Used to Answer Question 1

To answer this question, I used the `pit_stops` dataset because it records each pit stop made by each driver in each race. The key variable for measuring pit stop length is `milliseconds`, since it gives a consistent numeric measure of the duration of each stop.

First, I grouped the data by both `raceId` and `driverId` to calculate the average pit stop time for each driver in each race. Then, I grouped the data by `raceId` only in order to find the fastest and slowest pit stop in each race. The fastest pit stop is the minimum pit stop duration in that race, and the slowest pit stop is the maximum duration.

I converted the pit stop durations from milliseconds to seconds so that the final results would be easier to read and interpret.

In [0]:
from pyspark.sql.functions import avg, min, max, round, col

# 1. Average pit stop time for each driver in each race
avg_pit_per_driver_race = (
    df_pit
    .groupBy("raceId", "driverId")
    .agg(
        round(avg("milliseconds") / 1000, 3).alias("avg_pit_stop_seconds")
    )
)

# 2. Fastest and slowest pit stop in each race
race_pit_summary = (
    df_pit
    .groupBy("raceId")
    .agg(
        round(min("milliseconds") / 1000, 3).alias("fastest_pit_stop_seconds"),
        round(max("milliseconds") / 1000, 3).alias("slowest_pit_stop_seconds")
    )
)

# 3. Select driver names from drivers table
drivers_info = (
    df_drivers
    .select("driverId", "forename", "surname")
)

# 4. Combine the results and add driver names
q1_result = (
    avg_pit_per_driver_race
    .join(race_pit_summary, on="raceId", how="left")
    .join(drivers_info, on="driverId", how="left")
    .select(
        "raceId",
        "driverId",
        "forename",
        "surname",
        "avg_pit_stop_seconds",
        "fastest_pit_stop_seconds",
        "slowest_pit_stop_seconds"
    )
    .orderBy("raceId", "avg_pit_stop_seconds")
)

display(q1_result)

In [0]:
q1_result.show(20, truncate=False)

######## code explanation
df_pit.groupBy("raceId", "driverId"):
Group the pit stop data by race and driver, so each driver’s pit stops in each race are analyzed separately.

avg("milliseconds") / 1000:
Calculate the average pit stop time for each driver in each race, and convert the unit from milliseconds to seconds.

round(..., 3).alias("avg_pit_stop_seconds"):
Round the average pit stop time to three decimal places and rename the new column as avg_pit_stop_seconds.

df_pit.groupBy("raceId"):
Group the pit stop data by race to summarize pit stop performance at the race level.

min("milliseconds") / 1000:
Find the fastest pit stop in each race and convert it to seconds.

max("milliseconds") / 1000:
Find the slowest pit stop in each race and convert it to seconds.

df_drivers.select("driverId", "forename", "surname"):
Select driver identification and name columns from the drivers table for later merging.

.join(race_pit_summary, on="raceId", how="left"):
Merge the driver-level average pit stop results with the race-level fastest and slowest pit stop summary.

.join(drivers_info, on="driverId", how="left"):
Merge driver names into the result using driverId.

.select(...):
Keep only the required columns for the final output.

.orderBy("raceId", "avg_pit_stop_seconds"):
Sort the result by race and average pit stop time, so drivers with shorter average pit stop times appear first within each race.

### Problem2:  Rank order by finishing position the average time spent at the pit stop in each race.

In [0]:
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)

In [0]:
display(df_results)

####### Logic Used to Answer Question 2

For Question 2, I built on the result from Question 1, where I had already calculated the average pit stop time for each driver in each race. The new task here was not to recalculate pit stop averages, but to rank those averages according to each driver’s finishing position in the same race.

To do this, I used the `results` dataset, which contains the finishing order for each driver in each race. I joined the average pit stop table with the `results` table using `raceId` and `driverId`, since these two variables identify the same driver in the same race across both datasets. I then used `positionOrder` as the finishing position variable, because it is a numeric and consistent measure of final race ranking.

After joining the two datasets, I sorted the results by `raceId` and `positionOrder`. This produced a table showing, for each race, the drivers in finishing order together with their average pit stop time. This makes it possible to compare pit stop efficiency with final race outcome, even though the driver with the shortest average pit stop time is not necessarily the race winner.

In [0]:
# use q1 result and add finishing position
q2_result = (
    q1_result
    .join(df_results_q2, on=["raceId", "driverId"], how="left")
    .select(
        "raceId",
        "driverId",
        "forename",
        "surname",
        "positionOrder",
        "avg_pit_stop_seconds"
    )
    .orderBy("raceId", "positionOrder")
)

display(q2_result)

In [0]:
q2_result.show(20, truncate=False)

######## code explanation
join(df_results_q2, on=["raceId", "driverId"], how="left"):
Join the Q1 result table with the race results table by raceId and driverId to add each driver’s finishing position.

.select("raceId", "driverId", "forename", "surname", "positionOrder", "avg_pit_stop_seconds"):
Select only the relevant columns for the final output, including race ID, driver ID, driver names, finishing position, and average pit stop time.

"positionOrder":
This column represents the driver’s final finishing position in that race.

"avg_pit_stop_seconds":
This column keeps the average pit stop time calculated in Question 1.

.orderBy("raceId", "positionOrder"):
Sort the result first by race, then by finishing position, so drivers appear in their race ranking order.

### problem3: Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)

In [0]:
display(df_drivers)

####### Logic Used to Answer Question 3
To answer Question 3, I used only the `drivers` dataset. I first examined the existing values in the `code` column and compared them with drivers’ surnames. From this pattern, I found that the driver code is generally formed from the first three letters of the surname in uppercase. For example, Hamilton corresponds to `HAM`, Alonso to `ALO`, and Rosberg to `ROS`.

Based on this pattern, I filled in missing values in the original `code` column by using the driver’s surname. Specifically, when a code was missing, I removed any non-letter characters from the surname, took the first three letters, and converted them to uppercase. This was done to make sure that names containing punctuation or special characters would still produce a clean three-letter code.

I updated the original `code` column directly rather than creating a separate result table, so that the completed `drivers` table preserved all original variables while replacing only the missing code values.

In [0]:
from pyspark.sql.functions import col, when, upper, substring, regexp_replace

df_drivers = df_drivers.withColumn(
    "code",
    when(
        col("code").isNull() | (col("code") == r"\N"),
        upper(substring(regexp_replace(col("surname"), "[^A-Za-z]", ""), 1, 3))
    ).otherwise(col("code"))
)

display(df_drivers)

In [0]:
df_drivers.show(20, truncate=False)

######## code explanation
.withColumn("code", ...):
Update the existing code column in the df_drivers table.

when(...):
Apply a condition to check whether the current code value is missing or invalid.

col("code").isNull(): 
Identify rows where the code value is null.

(col("code") == r"\N"): 
Identify rows where the code contains \N, which is used in the dataset as a missing-value marker.

regexp_replace(col("surname"), "[^A-Za-z]", ""): 
Remove any non-letter characters from the driver’s surname before creating a new code.

substring(..., 1, 3):
Extract the first three letters of the cleaned surname.

upper(...):
Convert the extracted letters to uppercase so the generated code follows the standard driver code format.

.otherwise(col("code")):
Keep the original code value if it is already valid.

### Problem4: Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

In [0]:
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)

In [0]:
display(df_races)

######## Logic Used to Answer Question 4
The logic of Question 4 is to calculate each driver’s age at the time of each race, rather than using their current age or only comparing birth years. To do this, the results table is first used to identify which driver participated in each race. Then, the drivers table is joined to obtain each driver’s date of birth, and the races table is joined to obtain the race date. After these tables are combined, a new column called Age is created to represent the driver’s age on the date of the race. This makes it possible to compare drivers within the same race and identify the youngest and oldest driver in each race.


In [0]:
from pyspark.sql.functions import col, floor, months_between
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Step 1: keep the race-driver pairs from the results table
base_df = df_results.select(
    "raceId",
    "driverId"
)

# Step 2: get driver information from the drivers table
drivers_df_clean = df_drivers.select(
    "driverId",
    "forename",
    "surname",
    "dob"
)

# Step 3: get race information from the races table
races_df_clean = df_races.select(
    "raceId",
    col("date").alias("race_date"),
    col("name").alias("race_name")
)

# Step 4: join the tables and create the Age column
df_q4 = base_df \
    .join(drivers_df_clean, on="driverId", how="left") \
    .join(races_df_clean, on="raceId", how="left") \
    .withColumn("Age", floor(months_between(col("race_date"), col("dob")) / 12))

display(df_q4)

In [0]:
df_q4 = df_q4.orderBy("race_date", "raceId", "Age")
display(df_q4)

In [0]:
df_q4.show(20, truncate=False)

######## code explanation
df_results.select("raceId", "driverId"):
Keep only the race ID and driver ID from the results table, so the analysis starts with all race-driver combinations.

base_df:
Create a base table that identifies which drivers participated in which races.

df_drivers.select("driverId", "forename", "surname", "dob"):
Select driver information from the drivers table, including first name, surname, and date of birth.

drivers_df_clean:
Create a cleaner driver table with only the columns needed for the age calculation.

df_races.select("raceId", col("date").alias("race_date"), col("name").alias("race_name")):
Select race information from the races table and rename the columns for clarity.

alias("race_date"):
Rename the race date column to race_date so it is easier to distinguish from other date fields.

alias("race_name"):
Rename the race name column to race_name for a clearer final output.

.join(drivers_df_clean, on="driverId", how="left"):
Join the base table with the driver table to add each driver’s name and date of birth.

.join(races_df_clean, on="raceId", how="left"):
Join the result with the race table to add the race date and race name.

.withColumn("Age", ...):
Create a new column called Age to show the driver’s age at the time of each race.

months_between(col("race_date"), col("dob")) / 12:
Calculate the time difference between the race date and the driver’s date of birth in years.

floor(...):
Round the age down to the nearest whole number, so age is defined as the driver’s completed years on the race date.

### Problem5:  At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

######## logic to solve problem 5
To answer Question 5, the goal is to measure each driver’s cumulative podium record at the time of each race, rather than their total career podiums at the end of the dataset.

I started from the results table because it contains each driver’s finishing position in every race. Then I joined the races table to obtain the race date and race name, and joined the drivers table to add driver names. This creates one table that shows each driver’s race result together with the race information.

Next, I created three indicator variables: one for a win (positionOrder = 1), one for a second-place finish (positionOrder = 2), and one for a third-place finish (positionOrder = 3). These indicators make it possible to count each type of podium separately.
After that, I used a window function partitioned by driverId and ordered by race_date. This allows the code to look at each driver’s results in chronological order and calculate running totals over time.

Finally, I applied cumulative sums to the three indicator columns to create the new columns num_wins, num_2nd_places, and num_3rd_places. Each row therefore shows how many first-place, second-place, and third-place finishes that driver had up to and including that race.

In this question, “at any given race” is defined as the driver’s podium counts as of that race date, including the result of the current race itself.

In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum
from pyspark.sql.window import Window

# Step 1: join results with race information and driver information
df_q5 = df_results.select(
    "raceId",
    "driverId",
    "positionOrder"
).join(
    df_races.select(
        "raceId",
        col("date").alias("race_date"),
        col("name").alias("race_name")
    ),
    on="raceId",
    how="left"
).join(
    df_drivers.select(
        "driverId",
        "forename",
        "surname"
    ),
    on="driverId",
    how="left"
)

# Step 2: create indicator columns for 1st, 2nd, and 3rd place
df_q5 = df_q5 \
    .withColumn("win_flag", when(col("positionOrder") == 1, 1).otherwise(0)) \
    .withColumn("second_flag", when(col("positionOrder") == 2, 1).otherwise(0)) \
    .withColumn("third_flag", when(col("positionOrder") == 3, 1).otherwise(0))

# Step 3: define cumulative window including the current race
driver_window = Window.partitionBy("driverId").orderBy("race_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Step 4: create cumulative podium columns
df_q5 = df_q5 \
    .withColumn("num_wins", spark_sum("win_flag").over(driver_window)) \
    .withColumn("num_2nd_places", spark_sum("second_flag").over(driver_window)) \
    .withColumn("num_3rd_places", spark_sum("third_flag").over(driver_window))

# Step 5: select final columns
df_q5 = df_q5.select(
    "raceId",
    "race_name",
    "race_date",
    "driverId",
    "forename",
    "surname",
    "positionOrder",
    "num_wins",
    "num_2nd_places",
    "num_3rd_places"
)

display(df_q5)

In [0]:
df_q5 = df_q5.orderBy("race_date", "raceId", "positionOrder")
display(df_q5)

In [0]:
df_q5.show(20, truncate=False)

######## code explanation
df_results.select("raceId", "driverId", "positionOrder")
Select the race ID, driver ID, and finishing position from the results table as the core data for podium counting.

.join(df_races.select(...), on="raceId", how="left"):
Join the race information table to add the race date and race name for each result.

col("date").alias("race_date"):
Rename the race date column to race_date so it can be clearly used for chronological ordering.

col("name").alias("race_name"):
Rename the race name column to race_name for a clearer final output.

.join(df_drivers.select("driverId", "forename", "surname"), on="driverId", how="left"):
Join the drivers table to add each driver’s first name and surname.

.withColumn("win_flag", when(col("positionOrder") == 1, 1).otherwise(0)):
Create an indicator column that marks whether the driver won the race.

Window.partitionBy("driverId").orderBy("race_date"):
Define a window for each driver and sort their race records in chronological order.

.rowsBetween(Window.unboundedPreceding, Window.currentRow):
Set the window range so that each row includes all previous races and the current race for that driver.

spark_sum("win_flag").over(driver_window):
Calculate the cumulative number of wins for each driver up to and including the current race.

.select("raceId", "race_name", "race_date", "driverId", "forename", "surname", "positionOrder", "num_wins", "num_2nd_places", "num_3rd_places"):
Keep only the relevant columns for the final output.

### problem6: Continue exploring the data by answering your own question.
Is there a relationship between starting grid position and finishing position in each race?

######## Problem logic
I used the results dataset because it contains both grid (starting position) and positionOrder (final finishing position).
The idea is simple: if drivers start closer to the front, they may also be more likely to finish in better positions.

To examine this, I selected the relevant variables from the results table and kept only valid observations where both starting position and finishing position were available.

Then I sorted the output by race and starting position so that it is easy to compare where each driver started and where they finished.

This question provides a simple way to explore whether race performance is associated with grid advantage.

In [0]:
from pyspark.sql.functions import col

# Step 1: select relevant columns from results
base_q6 = df_results.select(
    "raceId",
    "driverId",
    "grid",
    "positionOrder"
)

# Step 2: add driver names
df_q6 = base_q6.join(
    df_drivers.select("driverId", "forename", "surname"),
    on="driverId",
    how="left"
)

# Step 3: keep only valid starting and finishing positions
df_q6 = df_q6.filter(
    (col("grid").isNotNull()) &
    (col("positionOrder").isNotNull()) &
    (col("grid") > 0) &
    (col("positionOrder") > 0)
)

# Step 4: sort the result for easier comparison
df_q6 = df_q6.orderBy("raceId", "grid")

display(df_q6)

In [0]:
df_q6.show(20, truncate=False)

######## code explanation
.join(df_drivers.select("driverId", "forename", "surname"), on="driverId", how="left"):
Join the drivers table to add each driver’s first name and surname.

on="driverId":
Use driverId as the matching key to connect driver information to the race results.

how="left":
Keep all rows from the base results table, even if some driver name information is missing.

.filter((col("grid").isNotNull()) & (col("positionOrder").isNotNull()) & (col("grid") > 0) & (col("positionOrder") > 0)):
Keep only valid observations where both starting position and finishing position are available and greater than zero.

col("grid").isNotNull():
Remove rows where the starting position is missing.

col("positionOrder").isNotNull():
Remove rows where the finishing position is missing.

(col("grid") > 0):
Keep only meaningful starting positions.

(col("positionOrder") > 0):
Keep only meaningful finishing positions.

.orderBy("raceId", "grid"):
Sort the data by race and starting position so it is easier to compare where drivers started and where they eventually finished.